In [ ]:
import json
from sklearn.model_selection import train_test_split
import re
from tqdm import tqdm
from datasets import Dataset
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import shap
import pickle
import string
from collections import defaultdict
import numpy as np
from glob import glob

In [ ]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

In [ ]:
from data_gathering.utils.clean_text import clean_text
from classification.utils_finetune import load_dataset, split_dataset

In [ ]:
import logging
logging.getLogger('shap').setLevel(logging.WARNING) # turns off the "shap INFO" logs
logging.getLogger('matplotlib').setLevel(logging.WARNING) # turns off the progress bar

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
with open(project_root + "/params.json", 'r') as f:
    PARAMS = json.load(f)

In [ ]:
model_id = PARAMS["classi_finetune_model"]
dataset = load_dataset()
print(f"Size of dataset: {len(dataset)}")
labels = list(dataset["label"].unique())

label2id, id2label, train_data, test_data, val_data = split_dataset(dataset, labels, train_size=PARAMS["train_split"], val_size=PARAMS["val_split"])

print(f'Train: {Counter(train_data["label"])}')
print(f'Test: {Counter(test_data["label"])}')
print(f'Val: {Counter(val_data["label"])}')

In [ ]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [ ]:
for l in dataset["label"].unique():
    print(l)
    subset = dataset[dataset["label"]==l]
    all_texts = " ".join(subset["text"])
    all_texts = all_texts.lower()
    all_texts = all_texts.translate(str.maketrans('', '', string.punctuation))
    all_texts = all_texts.split(" ")
    all_texts = [w for w in all_texts if w not in stop_words and w!=""]
    counter = Counter(all_texts)
    print(counter.most_common(20))
    print()

In [ ]:
dataset["label"].value_counts()

In [ ]:
len(dataset)

In [ ]:
val_data.to_pandas().to_csv("val_data.csv")

In [ ]:
# Load your fine-tuned BERT model and tokenizer
model_path = f'{PARAMS["save_model"]}{model_id.split("/")[-1]}_finetuned_16_12'
model_base = PARAMS["classi_finetune_model"]
tokenizer = AutoTokenizer.from_pretrained(model_base)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model = model.to("cuda")
model.eval()

In [ ]:
preds = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
)


In [ ]:
explainer = shap.Explainer(
    preds, seed=42
)

In [ ]:
texts = val_data["text"]

In [ ]:
size = 10

In [ ]:
# get shap values in batches

# shap_values = []
# for itr in range(0, len(texts), size):
#     print(f"Processing batch {itr} - {itr+size}")
#     batch_texts = texts[itr:itr+size]
#     batch_shap_values = explainer(batch_texts)
#     with open(f"shap_values_val/shap_batch_{itr}.pkl", "wb") as f:
#         pickle.dump(batch_shap_values, f)
#     shap_values.append(batch_shap_values)

In [ ]:
def get_sorted_shap_files(path_pattern="shap_values_val/shap_batch_*.pkl"):
    files = glob(path_pattern)

    def extract_number(f):
        match = re.search(r"shap_batch_(\d+)\.pkl", f)
        return int(match.group(1)) if match else -1

    return sorted(files, key=extract_number)

In [ ]:
files = get_sorted_shap_files()

In [ ]:

all_data = []
all_class_shap = []

for i in files:
    with open(i, "rb") as f:
        batch_sv = pickle.load(f)
    all_class_shap.extend(batch_sv.values)

# to keep the _ token (to join words back)            
for itr in range(0, len(texts), size):
    batch_texts = texts[itr:itr+size]
    for x in batch_texts:
        all_data.append(tokenizer.tokenize(x))


In [ ]:
len(all_class_shap), len(all_data), len(texts)

In [ ]:
def merge_tokens(tokens, values):
    merged_tokens = []
    merged_values = []

    current_token = ""
    current_value = None

    for t, v in zip(tokens, values):

        # RoBERTa word start token
        if t.startswith("▁") and t not in string.punctuation:

            # flush previous token
            if current_token != "":
                merged_tokens.append(current_token)
                merged_values.append(current_value)

            current_token = t[1:]  # remove _
            current_value = v.copy()

        else:
            if t not in string.punctuation:
                # continuation of same word
                current_token += t
                current_value += v

    # flush last token
    if current_token:
        merged_tokens.append(current_token)
        merged_values.append(current_value)

    return merged_tokens, np.array(merged_values)

In [ ]:
def group_values(tokens, values, group=2):
    paired_tokens = []
    paired_values = []

    for i in range(0, len(tokens), group):
        pair_tokens = tokens[i-group:i+group]
        pair_values = values[i-group:i+group]

        # join tokens with space
        paired_tokens.append(" ".join(pair_tokens))

        # combine values (sum, mean, etc.)
        paired_values.append(np.mean(pair_values, axis=0))

    return paired_tokens, np.array(paired_values)

In [ ]:
class_contribs = [defaultdict(list) for _ in range(3)]
class_contexts = [defaultdict(list) for _ in range(3)]
group = 2
for d, sv in zip(all_data, all_class_shap):
    merged_tokens, merged_values = merge_tokens(d, sv)
    merged_tokens, merged_values = group_values(merged_tokens, merged_values, group)
    for i, token in enumerate(merged_tokens):
        for c in range(3):
            # get context around max value words 
            # todo, fix error in context printing
            context = " ".join(
                            merged_tokens[max(i - group, 0): i]
                            + [f"[[{merged_tokens[i]}]]"]
                            + merged_tokens[i + 1: i + group + 1]
                    )
            
            class_contribs[c][token].append(merged_values[i, c])
            class_contexts[c][token].append(context)          
            

In [ ]:
len(class_contribs[0])

In [ ]:
top_k = 10

agg = []

for c in range(3):
    token_scores = {
        token: np.mean(vals)
        for token, vals in class_contribs[c].items()
    }
    agg.append(token_scores)
    

for c in range(3):

    print(f"\nTop words for class {id2label[c]}:")

    sorted_tokens = sorted(
        [(token, score) for token, score in agg[c].items() if score > 0],
        key=lambda x: x[1],
        reverse=True
    )

    for token, score in sorted_tokens[:top_k]:

        # choose one example context
        example_context = class_contexts[c][token][:3]

        print(f"\n{token}: {score:.4f}")
        examples = "\n    ".join(example_context)
        print(f"    {examples}")

In [ ]:
top_k = 10

agg = []

for c in range(3):
    token_scores = {
        token: np.mean(vals)
        for token, vals in class_contribs[c].items()
    }
    agg.append(token_scores)
    

for c in range(3):

    print(f"\nTop words for class {id2label[c]}:")

    sorted_tokens = sorted(
        [(token, score) for token, score in agg[c].items() if score < 0],
        key=lambda x: x[1],
        reverse=False
    )

    for token, score in sorted_tokens[:top_k]:

        # choose one example context
        example_context = class_contexts[c][token][:3]

        print(f"\n{token}: {score:.4f}")
        examples = "\n    ".join(example_context)
        print(f"    {examples}")